# Clase 7 · ¿Cómo lee una máquina un documento?

Imaginen esto: tienen 300 actas escaneadas de una junta parroquial y quieren encontrar todas las que mencionan la palabra "agua". Abren el PDF, prueban Ctrl+F... y no encuentra nada.

¿Por qué? Porque un PDF escaneado **es una foto**. Para la computadora no hay letras, solo píxeles, igual que en las fotos de la semana pasada.

El **OCR** (reconocimiento óptico de caracteres) es el modelo que convierte esa foto en texto.

**Nuestro documento:** el programa de seminarios académicos de la VII Bienal de Arquitectura de Quito (BAQ), de 1990. Son cuatro páginas escritas a máquina, escaneadas, con correcciones a mano: nombres tachados, nombres agregados con esfero, visto buenos. Es decir, un documento de archivo real.

**Pregunta guía:** ¿qué se vuelve texto y qué se pierde en el camino?

## 1. Preparar el entorno

Necesitamos dos librerías que no vienen en Colab:

- `easyocr`: el modelo de OCR.
- `pymupdf`: abre archivos PDF y convierte sus páginas en imágenes.

Recuerden activar la GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. Sin GPU también funciona, pero más lento.

In [ ]:
!pip install -q easyocr pymupdf

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import pymupdf
import easyocr

## 2. Abrir el PDF

Primero descargamos el PDF del repositorio del curso y lo guardamos en Colab.

In [ ]:
enlace = "https://raw.githubusercontent.com/DiversaStudio/Inteligencia-Artificial-con-python-FLACSO/main/datos/documentos/BAQ1990_seminarios.pdf"

respuesta = requests.get(enlace)
with open("BAQ1990_seminarios.pdf", "wb") as archivo:
    archivo.write(respuesta.content)

**Línea por línea**
- `requests.get(enlace)` descarga el archivo.
- `open(..., "wb")` crea un archivo nuevo en Colab. `"wb"` significa escribir (*write*) en modo binario, porque un PDF no es texto plano.
- `archivo.write(respuesta.content)` guarda lo que descargamos dentro de ese archivo.

Si quieren usar un PDF propio, súbanlo con el ícono de carpeta de la izquierda y cambien el nombre en la siguiente celda.

In [ ]:
documento = pymupdf.open("BAQ1990_seminarios.pdf")
print(len(documento))

`len(documento)` nos dice cuántas páginas tiene: 4.

Un documento de `pymupdf` se comporta como una lista de páginas. Igual que con las listas, empezamos a contar desde 0: `documento[0]` es la primera página.

### Convertir una página en imagen

El OCR no lee PDF, lee imágenes. Así que convertimos la página en una foto:

In [ ]:
pagina = documento[0]
foto = pagina.get_pixmap(dpi=150)
imagen = Image.frombytes("RGB", [foto.width, foto.height], foto.samples)

print(imagen.size)
imagen

**Línea por línea**
- `pagina.get_pixmap(dpi=150)` "fotografía" la página. `dpi` es la resolución: cuántos puntos por pulgada. Más dpi es más detalle, pero también más lento.
- `Image.frombytes(...)` convierte esa foto en una imagen de PIL, la misma librería que usamos con las fotos de Street View.
- `imagen.size` dice el tamaño en píxeles: (ancho, alto).

Miren la página con calma. ¿Qué ven además del texto a máquina?

### Tu turno

Convierte la **cuarta página** en imagen y muéstrala. Guárdala en una variable llamada `imagen_4`.

Pista: copia la celda de arriba y cambia solo lo necesario. ¿Qué número tiene la cuarta página?

In [ ]:
# escribe aquí

## 3. Leer la página con OCR

Cargamos el modelo. Le decimos en qué idioma está el documento: `"es"` es español. La primera vez descarga el modelo, así que tarda un poco.

In [ ]:
lector = easyocr.Reader(["es"])

In [ ]:
resultados = lector.readtext(np.array(imagen))
print(len(resultados))

**Línea por línea**
- `np.array(imagen)` convierte la imagen en la tabla de números que vimos en la clase 4. El modelo trabaja con números, no con fotos.
- `lector.readtext(...)` busca texto en la imagen y lo lee.
- `len(resultados)` dice cuántos pedazos de texto encontró.

Veamos el primer resultado:

In [ ]:
resultados[0]

Cada resultado tiene **tres partes**, igual que las detecciones de YOLO:

| YOLO (clase 5) | EasyOCR (hoy) |
|---|---|
| una caja | una caja (las cuatro esquinas del texto) |
| una etiqueta: "car", "person" | una etiqueta: la palabra que leyó |
| una confianza | una confianza |

La diferencia es que YOLO elegía entre 80 categorías y el OCR elige entre letras. Pero la lógica es la misma: **el modelo no "lee", calcula qué letras se parecen más a esos píxeles**.

Podemos separar las tres partes así:

In [ ]:
caja, texto, confianza = resultados[0]

print(texto)
print(confianza)

### Tu turno

Escribe un `for` que recorra **todos** los resultados y muestre, en cada vuelta, el texto y la confianza redondeada a dos decimales.

Pista: es la misma línea `caja, texto, confianza = ...`, pero dentro del `for`. Para redondear se usa `round(numero, 2)`.

In [ ]:
# escribe aquí

**Para mirar en la salida:**
- ¿Qué confianza tienen los nombres escritos a máquina?
- Busquen el nombre que alguien agregó a mano en el subtema 4 (con esfero azul, al lado de "ARQ."). ¿Qué leyó el modelo? ¿Con qué confianza?

## 4. Ver lo que el modelo leyó y lo que no

Una lista de palabras no deja ver **dónde** se equivoca el modelo. Esta función dibuja las cajas sobre la página: **verde** si la confianza es alta y **roja** si es baja.

No hace falta que entiendan cada línea de la función. Lo importante es cómo se usa.

In [ ]:
def dibujar_lectura(imagen, resultados, umbral=0.5):
    copia = imagen.copy()
    lapiz = ImageDraw.Draw(copia)
    for caja, texto, confianza in resultados:
        esquinas = [tuple(punto) for punto in caja]
        if confianza >= umbral:
            color = "green"
        else:
            color = "red"
        lapiz.polygon(esquinas, outline=color, width=3)
    return copia

In [ ]:
dibujar_lectura(imagen, resultados)

**Para comentar:**
- ¿Qué tienen en común las cajas rojas?
- ¿Hay partes de la página que **no tienen ninguna caja**? (miren los visto buenos, los números de página, los círculos)
- Si alguien cambia `umbral=0.5` por `umbral=0.8`, ¿qué creen que pasa?

### Tu turno

Muestra solo las palabras **dudosas**: las que tienen una confianza menor a 0.4.

Pista: es el `for` de la sección anterior, con un `if` adentro.

In [ ]:
# escribe aquí

## 5. Todo el documento: una ficha por palabra

Ahora leemos las cuatro páginas. Usamos el patrón de siempre: **recorrer, guardar una ficha en cada vuelta y armar una tabla al final**.

Aquí hay dos `for`, uno dentro del otro, como en el censo de Street View:
- el primero recorre las **páginas**;
- el segundo recorre los **textos** que el modelo encontró en cada página.

Esta celda tarda: el modelo lee las cuatro páginas.

In [ ]:
fichas = []

for numero, pagina in enumerate(documento):
    foto = pagina.get_pixmap(dpi=150)
    imagen_pagina = Image.frombytes("RGB", [foto.width, foto.height], foto.samples)
    resultados_pagina = lector.readtext(np.array(imagen_pagina))

    for caja, texto, confianza in resultados_pagina:
        ficha = {
            "pagina": numero + 1,
            "texto": texto,
            "confianza": round(confianza, 2),
        }
        fichas.append(ficha)

lecturas = pd.DataFrame(fichas)
lecturas.head()

**Línea por línea**
- `enumerate(documento)` recorre las páginas y además nos da su número (0, 1, 2, 3). Por eso guardamos `numero + 1`: para que la primera página sea la 1 y no la 0.
- El `for` de adentro es el mismo de la sección 3, pero ahora cada texto se guarda como una ficha.
- `pd.DataFrame(fichas)` convierte la lista de fichas en una tabla.

### Guardar el texto

Juntamos todo en un solo texto y lo guardamos en un archivo `.txt`. Ese archivo es el que vamos a analizar el lunes.

In [ ]:
texto_completo = " ".join(lecturas["texto"])

with open("BAQ1990_texto.txt", "w") as archivo:
    archivo.write(texto_completo)

print(texto_completo[:500])

`" ".join(...)` pega todos los textos de la columna, separados por un espacio. `[:500]` muestra solo los primeros 500 caracteres.

Ahora sí podemos hacer Ctrl+F. Pero, ¿es el mismo documento?

### Tu turno

Calcula la **confianza promedio de cada página**.

Pista: lo hicimos con las entrevistas. Hay que agrupar por `"pagina"` y calcular el promedio (`mean`) de `"confianza"`.

In [ ]:
# escribe aquí

**Para comentar:** ¿por qué la confianza baja de la página 1 a la página 4? Vuelvan a mirar las páginas.

## 6. Primer hallazgo: ¿de dónde venían los expositores?

Con el texto ya podemos hacer una pregunta de investigación. Por ejemplo: ¿qué países estaban representados en los seminarios de la Bienal?

Una forma sencilla es contar cuántas veces aparece el nombre de cada país en el texto. Primero pasamos todo a mayúsculas para que "Ecuador" y "ECUADOR" cuenten igual.

In [ ]:
texto_mayusculas = texto_completo.upper()

print(texto_mayusculas.count("ECUADOR"))
print(texto_mayusculas.count("CHILE"))

`.count("ECUADOR")` cuenta cuántas veces aparece esa palabra en el texto.

### Tu turno

1. Crea una lista llamada `paises` con al menos seis países: Ecuador, Chile, Argentina, México, Colombia, Uruguay, Costa Rica...
2. Recorre la lista con un `for` y muestra cuántas veces aparece cada país.

Cuidado: el documento está escrito a máquina, sin tildes. Escriban `MEXICO` y `PERU`, no `MÉXICO` ni `PERÚ`.

In [ ]:
# escribe aquí

### Ahora, la pregunta importante

El conteo dice que **Costa Rica aparece una vez**. Vuelvan al PDF y busquen "Costa Rica" a ojo, en las cuatro páginas.

- ¿Cuántas veces aparece en realidad?
- ¿Cuál de esas apariciones contó el modelo?
- ¿Esa persona participó en la Bienal o fue tachada?

Pista: la que contó el modelo está en la página 2.

## Cierre

**Lo que hicimos hoy:** PDF escaneado → imagen → OCR → tabla de fichas → texto buscable → primer conteo.

**Lo que aprendimos del documento:**
- El OCR lee muy bien lo que se parece a sus datos de entrenamiento: texto impreso, limpio, horizontal.
- Lo que no se parece (letra a mano, tachones, sellos, visto buenos) se pierde o, peor, se lee mal sin avisar.
- En este documento, **lo escrito a mano es justamente la información más reciente**: quién se cambió, quién no vino, quién se agregó. El OCR conserva la versión vieja y borra las decisiones.

**Para pensar:**
1. Si un archivo histórico se digitaliza solo con OCR, ¿qué documentos quedan "buscables" y cuáles se vuelven invisibles? ¿Qué voces quedan en cada grupo?
2. ¿Quién debería revisar los errores? ¿Cuánto trabajo humano hay detrás de un archivo "digitalizado con IA"?
3. Si tuvieran que usar este texto en una investigación, ¿qué le dirían a su lector sobre cómo se obtuvo?

**Próxima clase:** trabajamos con el texto que sacamos hoy: buscar, limpiar, contar y encontrar temas.